# Real-Time Use Case 2 — Agentic AI for IT Incident Response

## Scenario

An enterprise application experiences production incidents such as:

- Login failures.
- Database CPU spikes.
- API latency.
- Memory exhaustion.
- DNS delays.
- Kafka backlog.

Instead of using one large agent for everything, we create a small **AI incident-response team**.

## Agents in this workflow

### 1. Triage Agent
Determines urgency, business impact, and what should be investigated first.

### 2. Diagnostic Agent
Analyzes the observed signal and proposes likely technical causes.

### 3. Remediation Agent
Creates a safe remediation plan.

### 4. Communication Agent
Creates a concise stakeholder update.

### 5. Supervisor / Coordinator
Coordinates the workflow and combines the specialist outputs.

## Why this is Agentic AI

This workflow demonstrates important agentic characteristics:

- Multiple specialized agents.
- Role-based collaboration.
- State passed between agents.
- Multi-step reasoning.
- Autonomous task decomposition.
- A controlled action boundary.
- Human approval before potentially disruptive remediation.

The flow is:

**Incident → Triage → Diagnosis → Remediation Plan → Human Approval → Stakeholder Communication → Final Incident Report**

This is more than a chatbot. It is a goal-oriented workflow involving multiple AI workers.

## Step 1 — Install packages

This demo uses the same OpenAI integration through `langchain_openai`.

The workflow is deliberately implemented in simple Python so participants can clearly see how the agents cooperate.

In [1]:
# Run once if required
# %pip install -U langchain langchain-openai python-dotenv pandas pydantic

## Step 2 — Imports and environment

The API key is loaded from `.env`.

Example:

```text
OPENAI_API_KEY=your_key_here
```

In [2]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

## Step 3 — Load the incident dataset

The incident CSV represents a simplified real-time incident queue.

In production, this information could come from:

- ServiceNow.
- PagerDuty.
- Datadog.
- Splunk.
- Elasticsearch.
- CloudWatch.
- Prometheus.
- Kubernetes APIs.

For training, the local CSV keeps the demonstration easy to reproduce.

In [3]:
incidents = pd.read_csv("agentic_it_incidents.csv")
display(incidents)
print("Incident count:", len(incidents))

,incident_id,title,service,severity,status,affected_users,observed_signal,detected_at
0,INC001,Payments API latency,payments-api,HIGH,Open,780,p95 latency 4.8s; DB pool saturation,2026-09-10 09:05
1,INC002,Login failures,auth-service,CRITICAL,Open,2400,HTTP 500 spike; token validation timeout,2026-09-10 09:18
2,INC003,Checkout pod restart,checkout-service,MEDIUM,Monitoring,320,OOMKilled; memory reached 95%,2026-09-10 08:47
3,INC004,Search slow queries,search-service,MEDIUM,Open,640,Elasticsearch query timeout > 3s,2026-09-10 09:22
4,INC005,Notification backlog,notification-worker,LOW,Open,150,Kafka lag 18200 messages,2026-09-10 09:28
5,INC006,Database CPU spike,orders-db,HIGH,Open,930,CPU 96%; top query missing index,2026-09-10 09:31
6,INC007,Image upload failures,media-service,MEDIUM,Resolved,410,Object store 503 transient errors,2026-09-10 07:55
7,INC008,DNS resolution delay,edge-gateway,HIGH,Open,1100,DNS lookup p95 2.1s,2026-09-10 09:36
8,INC009,Cache miss surge,catalog-cache,LOW,Monitoring,260,Redis eviction rate increased,2026-09-10 08:58
9,INC010,Payment webhook retries,payments-worker,HIGH,Open,720,Partner endpoint timeout; retries exhausted,2026-09-10 09:41


Incident count: 10


## Step 4 — Create controlled operational tools

These tools represent data sources or operational capabilities.

For safety, this demo does **not automatically restart services or change infrastructure**.

Instead, it generates a remediation proposal that must be approved by a human.

In [4]:
@tool
def get_incident(incident_id: str) -> str:
    """Retrieve the incident record for a given incident ID."""
    row = incidents[incidents["incident_id"].str.upper() == incident_id.upper()]
    if row.empty:
        return f"Incident {incident_id} was not found."
    return row.iloc[0].to_json()


@tool
def get_service_runbook(service: str) -> str:
    """Return a simplified operational runbook for a service."""
    runbooks = {
        "auth-service": (
            "Check token provider health; inspect authentication error rate; "
            "verify dependency latency; scale only after confirming saturation."
        ),
        "orders-db": (
            "Check CPU and active sessions; inspect top SQL; verify indexes; "
            "avoid restart unless approved by the incident commander."
        ),
        "payments-api": (
            "Check downstream dependencies; inspect latency percentiles; "
            "review DB pool and connection saturation."
        ),
        "checkout-service": (
            "Inspect pod memory; check OOMKilled events; compare memory request/limit; "
            "review recent deployment changes."
        )
    }
    return runbooks.get(
        service,
        "Collect logs, metrics, traces, recent deployment changes, dependency health, and rollback options."
    )


@tool
def get_recent_change(service: str) -> str:
    """Return a simulated recent deployment/change record for a service."""
    changes = {
        "auth-service": "Version 3.8.1 deployed 35 minutes before incident; token-validation timeout changed from 2s to 1s.",
        "orders-db": "New reporting query released this morning; no database restart or schema migration.",
        "payments-api": "Connection pool max size reduced from 120 to 80 during a configuration update.",
        "checkout-service": "Version 5.4.0 deployed 2 hours ago with a new image-processing dependency."
    }
    return changes.get(service, "No significant recent change found in the demo change log.")

## Step 5 — Create the shared LLM

All specialist agents can use the same underlying OpenAI model while receiving different system instructions.

This is common in multi-agent systems: specialization often comes from **role, tools, instructions, and accessible context**, not necessarily from using different LLMs.

In [5]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

## Step 6 — Create the Triage Agent

The Triage Agent focuses only on:

- Severity.
- Business impact.
- Immediate priority.
- Escalation recommendation.

It can retrieve the incident but should not attempt deep diagnosis.

In [6]:
triage_agent = create_agent(
    model=llm,
    tools=[get_incident],
    system_prompt="""
You are the Triage Agent in an enterprise incident-response team.

Your job:
- retrieve the incident,
- assess urgency,
- summarize business impact,
- identify the first investigation priority,
- recommend whether immediate escalation is required.

Do not invent facts.
Do not propose risky infrastructure actions.
Return a concise technical triage note.
"""
)

## Step 7 — Create the Diagnostic Agent

The Diagnostic Agent has access to:

- Incident data.
- Service runbook.
- Recent change information.

Its job is to form evidence-based hypotheses rather than immediately taking action.

In [7]:
diagnostic_agent = create_agent(
    model=llm,
    tools=[get_incident, get_service_runbook, get_recent_change],
    system_prompt="""
You are the Diagnostic Agent.

Analyze an IT incident using available tools.

Produce:
1. Most likely root-cause hypotheses.
2. Evidence supporting each hypothesis.
3. Additional checks needed.
4. The most likely cause.

Do not claim certainty unless the evidence supports it.
Do not execute remediation.
"""
)

## Step 8 — Create the Remediation Agent

The Remediation Agent receives the triage and diagnosis results.

Its purpose is to create a **safe proposed action plan**.

A key governance rule is included:

> No disruptive production action is automatically executed.

The final plan must explicitly identify actions requiring human approval.

In [8]:
remediation_agent = create_agent(
    model=llm,
    tools=[get_service_runbook],
    system_prompt="""
You are the Remediation Planning Agent.

Using the incident context, triage result, and diagnostic result:
- propose the safest remediation sequence,
- start with reversible and low-risk actions,
- identify validation checks,
- identify rollback steps,
- clearly mark any restart, scale, configuration change, database change,
  traffic shift, or rollback as HUMAN APPROVAL REQUIRED.

You are a planning agent only. Never claim that an action was executed.
"""
)

## Step 9 — Create the Communication Agent

Technical incident data is often too detailed for business stakeholders.

The Communication Agent converts the current incident state into a short update containing:

- What happened.
- Customer/business impact.
- What the technical team is doing.
- Whether the issue is resolved or still under investigation.

It must avoid unsupported ETAs.

In [9]:
communication_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are the Incident Communication Agent.

Create a short stakeholder update from the information provided.

Include:
- incident summary,
- customer/business impact,
- current investigation/remediation status,
- next update wording.

Do not invent an ETA.
Do not expose unnecessary internal technical details.
"""
)

## Step 10 — Helper function for reading each agent's final response

Every agent returns a message history.

This helper extracts only the final assistant message so it can be passed to the next agent.

In [10]:
def final_text(result):
    return result["messages"][-1].content

## Step 11 — Agentic workflow coordinator

This function acts as the workflow supervisor.

It coordinates the specialist agents in sequence and maintains shared state in a Python dictionary.

The key point is that the output of one agent becomes context for another agent.

For a training demo, this explicit implementation is useful because participants can clearly see every handoff.

In [11]:
def run_incident_response(incident_id: str, human_approved: bool = False):
    state = {
        "incident_id": incident_id,
        "human_approved": human_approved
    }

    # 1. TRIAGE
    triage_result = triage_agent.invoke({
        "messages": [{
            "role": "user",
            "content": f"Triage incident {incident_id}."
        }]
    })
    state["triage"] = final_text(triage_result)

    # 2. DIAGNOSIS
    diagnosis_result = diagnostic_agent.invoke({
        "messages": [{
            "role": "user",
            "content": (
                f"Investigate incident {incident_id}.\n\n"
                f"Triage note:\n{state['triage']}"
            )
        }]
    })
    state["diagnosis"] = final_text(diagnosis_result)

    # 3. REMEDIATION PLANNING
    remediation_result = remediation_agent.invoke({
        "messages": [{
            "role": "user",
            "content": (
                f"Create a remediation plan for incident {incident_id}.\n\n"
                f"TRIAGE:\n{state['triage']}\n\n"
                f"DIAGNOSIS:\n{state['diagnosis']}"
            )
        }]
    })
    state["remediation_plan"] = final_text(remediation_result)

    # 4. GOVERNANCE / HUMAN APPROVAL GATE
    if human_approved:
        state["action_status"] = (
            "Human approval received. The approved remediation may now be "
            "executed by an authorized operational system."
        )
    else:
        state["action_status"] = (
            "No production-changing action executed. "
            "Human approval is required before disruptive remediation."
        )

    # 5. COMMUNICATION
    communication_result = communication_agent.invoke({
        "messages": [{
            "role": "user",
            "content": (
                f"Create a stakeholder update for incident {incident_id}.\n\n"
                f"TRIAGE:\n{state['triage']}\n\n"
                f"DIAGNOSIS:\n{state['diagnosis']}\n\n"
                f"REMEDIATION PLAN:\n{state['remediation_plan']}\n\n"
                f"ACTION STATUS:\n{state['action_status']}"
            )
        }]
    })
    state["communication"] = final_text(communication_result)

    return state

## Step 12 — Run the Agentic AI workflow

We will investigate a critical login failure incident.

Set `human_approved=False` first.

This demonstrates the governance boundary: the AI system can reason, investigate, plan, and communicate, but it cannot claim to have changed production infrastructure.

In [12]:
state = run_incident_response(
    incident_id="INC002",
    human_approved=False
)

for key, value in state.items():
    print(f"\n{'='*20} {key.upper()} {'='*20}")
    print(value)


==================== INCIDENT_ID ====================
INC002

==================== HUMAN_APPROVED ====================
False

==================== TRIAGE ====================
Incident INC002 involves critical login failures in the auth-service, with a spike in HTTP 500 errors and token validation timeouts. It affects 2400 users, indicating a significant business impact by potentially blocking user access. The first investigation priority should be to analyze the auth-service logs and token validation processes to identify the root cause of the failures. Immediate escalation is recommended due to the critical severity and large user impact.

==================== DIAGNOSIS ====================
1. Most likely root-cause hypotheses:
   - Hypothesis A: The recent deployment of auth-service version 3.8.1 introduced a regression causing token validation timeouts and HTTP 500 errors.
   - Hypothesis B: The token provider or a dependent service is experiencing health or latency issues, causing

## Step 13 — Simulate the human approval decision

For demonstration purposes, change the flag to `True`.

This does **not** perform an actual production action. It only changes the workflow state to show what happens after authorization.

In a real system, the approved step could trigger:

- A ServiceNow change.
- A Kubernetes deployment API.
- A cloud automation workflow.
- A rollback pipeline.
- An infrastructure-as-code job.

Those integrations should use strict permissions and auditable controls.

In [13]:
approved_state = run_incident_response(
    incident_id="INC002",
    human_approved=True
)

print(approved_state["action_status"])
print("\nStakeholder communication:\n")
print(approved_state["communication"])

Human approval received. The approved remediation may now be executed by an authorized operational system.

Stakeholder communication:

**Incident Update: INC002 – Critical Login Failures in Auth-Service**

**Incident Summary:**  
We are currently experiencing critical login failures affecting approximately 2,400 users due to increased HTTP 500 errors and token validation timeouts in the authentication service. This issue began shortly after the deployment of auth-service version 3.8.1, which introduced a reduced token validation timeout.

**Customer/Business Impact:**  
Users are unable to authenticate successfully, resulting in significant disruption to access and service usability.

**Current Status:**  
Human approval has been obtained to proceed with the remediation plan. We are initiating detailed log and health analyses to confirm the root cause and will begin reverting the token validation timeout setting if necessary. All remediation steps will be carefully validated to ensure

## Step 14 — Try other incidents

Each incident produces a different collaboration path because the agents reason over different signals.

Suggested demonstrations:

- `INC001` — payments API latency.
- `INC003` — Kubernetes pod memory issue.
- `INC006` — database CPU spike.
- `INC008` — DNS delay.

The interesting part for participants is not only the final answer. It is the **handoff between specialized roles**.

In [14]:
for incident_id in ["INC001", "INC003", "INC006"]:
    result = run_incident_response(incident_id, human_approved=False)
    print("\n" + "#" * 70)
    print("INCIDENT:", incident_id)
    print("#" * 70)
    print("\nTRIAGE\n", result["triage"])
    print("\nDIAGNOSIS\n", result["diagnosis"])
    print("\nREMEDIATION\n", result["remediation_plan"])


######################################################################
INCIDENT: INC001
######################################################################

TRIAGE
 Incident INC001 involves high latency in the payments API, with a p95 latency of 4.8 seconds and database pool saturation observed. This affects 780 users and is currently open with high severity.

Urgency: High
Business Impact: Significant user experience degradation for payments, potentially impacting transaction success and revenue.
First Investigation Priority: Investigate database connection pool saturation and its impact on API latency.
Recommendation: Immediate escalation is recommended due to high severity and user impact.

DIAGNOSIS
 1. Most likely root-cause hypotheses:
   a. Database connection pool saturation is causing increased wait times for database queries, leading to high latency in the payments API.
   b. There may be an underlying database performance issue or query inefficiency exacerbating the conn

# What participants should observe

## Agent behavior

The Triage Agent is not expected to perform the Diagnostic Agent's job.

The Diagnostic Agent uses more technical tools and context.

The Remediation Agent receives earlier outputs and plans an action sequence.

The Communication Agent converts technical state into business-friendly language.

## Agentic AI characteristics demonstrated

1. **Goal-oriented behavior** — resolve or stabilize an incident.
2. **Decomposition** — the problem is split into triage, diagnosis, remediation, and communication.
3. **Specialization** — each agent has a clear role.
4. **Collaboration** — agents consume previous agents' outputs.
5. **State management** — the workflow stores triage, diagnosis, plan, approval, and communication.
6. **Tool use** — agents query incident records, runbooks, and change information.
7. **Autonomy** — the system progresses through multiple reasoning stages without the user manually prompting every agent.
8. **Governance** — risky actions are gated.
9. **Human-in-the-loop** — production changes require authorization.
10. **Auditability** — each intermediate output can be logged and inspected.

# Agent vs Agentic AI

| Area | Agent | Agentic AI |
|---|---|---|
| Number of reasoning roles | Usually one | Multiple or dynamically coordinated roles |
| Main behavior | Chooses tools and answers | Plans and executes a multi-step workflow |
| Coordination | Limited | Central concept |
| State | Often simple conversation/tool state | Shared workflow state |
| Delegation | Usually not required | Common |
| Human approval | Optional | Often important for high-impact actions |
| Example in this pack | E-commerce support agent | IT incident-response team |

# Production enhancements

A production version could add:

- LangGraph for durable workflow orchestration.
- Persistent checkpointing.
- LangSmith or Langfuse tracing.
- Evaluation datasets.
- Prompt-injection protection.
- PII masking.
- Role-based tool authorization.
- Change-management integration.
- Retry and timeout policies.
- Cost/token monitoring.
- SLA tracking.
- Human approval UI.
- Incident postmortem generation.

# Key teaching message

**An AI agent decides what tool to use.**

**Agentic AI coordinates multiple reasoning steps, roles, tools, state, and actions toward a larger goal.**
